In [86]:
import pandas as pd
import numpy as np
import re

In [87]:
df = pd.read_csv('online_retail_real_world.csv')
df.head()

,OrderID,CustomerID,ProductName,Brand,Raw_Weight,Country,OrderDate,UnitPrice
0,REC-861088,6120,Perly,Perly,100 g,"Morocco,United States",2024-02-22,13.18
1,REC-900952,12550,Prince Goût Chocolat,LU,300 g,"Algeria,Belgium,France,French Polynesia,German...",2024-11-13,1.50
2,REC-595965,6705,Excellence Noir Prodigieux 90% Cacao,"Lindt EXCELLENCE,Lindt",100g,"Argelia,Austria,Bélgica,Bulgaria,Canadá,Repúbl...",2025-02-02,12.99
3,REC-942653,8645,Tonik,عربي,22 g,Maroc,2024-12-11,14.82
4,REC-885998,14430,Sésame,Gerblé,230g,"Belgium, Bulgaria, France, en:switzerland",2025-08-21,18.40


In [88]:
print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist(),"\n")
df.info()

Shape: (3000, 8)

Columns: ['OrderID', 'CustomerID', 'ProductName', 'Brand', 'Raw_Weight', 'Country', 'OrderDate', 'UnitPrice'] 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3000 entries, 0 to 2999
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   OrderID      3000 non-null   object 
 1   CustomerID   3000 non-null   int64  
 2   ProductName  2904 non-null   object 
 3   Brand        2665 non-null   object 
 4   Raw_Weight   2488 non-null   object 
 5   Country      3000 non-null   object 
 6   OrderDate    3000 non-null   object 
 7   UnitPrice    2615 non-null   float64
dtypes: float64(1), int64(1), object(6)
memory usage: 187.6+ KB


In [89]:
print("Total duplicates:", df.duplicated().sum())
print("\nMissing values:\n", df.isnull().sum())

Total duplicates: 0

Missing values:
 OrderID          0
CustomerID       0
ProductName     96
Brand          335
Raw_Weight     512
Country          0
OrderDate        0
UnitPrice      385
dtype: int64


There is not duplicate values.

In [97]:
df['OrderDate'] = pd.to_datetime(df['OrderDate'], errors='coerce')
df['OrderDate'].dtype

dtype('<M8[ns]')

In [99]:
#data from 2024-02-05 to 2026-02-04
df['OrderDate'] = pd.to_datetime(df['OrderDate'], errors='coerce')
print("Date range:", df['OrderDate'].min(), "to", df['OrderDate'].max()) 

Date range: 2024-02-05 00:00:00 to 2026-02-04 00:00:00


In [100]:
#Clean Text Columns
#Handling missing values with 'unknown'
df['ProductName'] = df['ProductName'].fillna('unknown').str.lower().str.strip()

#Remove emojis and special characters
df['ProductName'] = df['ProductName'].apply(
    lambda x: re.sub(r'[\U0001F600-\U0001F64F\U0001F300-\U0001F5FF\U0001F680-\U0001F6FF\U0001F1E0-\U0001F1FF]', '', str(x))
)

df['Brand'] = df['Brand'].fillna('unknown').str.lower().str.strip()
df['Country'] = df['Country'].fillna('unknown').str.lower().str.strip()


In [101]:
#Clean Numeric Columns
df['UnitPrice'] = pd.to_numeric(df['UnitPrice'], errors='coerce')
#Fill missing prices with median
df['UnitPrice'] = df['UnitPrice'].fillna(df['UnitPrice'].median())
#Check for negative or zero prices
print("Negative or zero prices:", (df['UnitPrice'] <= 0).sum())

Negative or zero prices: 0


Convert weight strings to grams.
Examples:
    "500g" -> 500
    "1 kg" -> 1000
    "700" -> 700


In [ ]:
def convert_to_grams(w):
    if pd.isna(w) or str(w).strip() == '':
        return np.nan
    w = str(w).lower().strip().replace('klg', 'kg').replace('gram', 'g')
    # Extract number
    num_match = re.search(r'([\d.]+)', w)
    if not num_match:
        return np.nan
    value = float(num_match.group(1))
    
    # Detect unit
    if 'kg' in w:
        return value * 1000
    elif 'ml' in w or 'l' in w: 
        return value * (1000 if 'l' in w else 1)
    else:
        return value  # default = grams

df['Weight_g'] = df['Raw_Weight'].apply(convert_to_grams)
df['Weight_g'] = df['Weight_g'].fillna(df['Weight_g'].median())  #Fill missing weights with median
df = df.drop(columns=['Raw_Weight'])  # we now have clean numeric weight

In [104]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3000 entries, 0 to 2999
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   OrderID      3000 non-null   object        
 1   CustomerID   3000 non-null   int64         
 2   ProductName  3000 non-null   object        
 3   Brand        3000 non-null   object        
 4   Raw_Weight   3000 non-null   float64       
 5   Country      3000 non-null   object        
 6   OrderDate    3000 non-null   datetime64[ns]
 7   UnitPrice    3000 non-null   float64       
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 187.6+ KB


In [103]:
print("\nSample of cleaned data:")
df.head()


Sample of cleaned data:


,OrderID,CustomerID,ProductName,Brand,Raw_Weight,Country,OrderDate,UnitPrice
0,REC-861088,6120,perly,perly,100.0,"morocco,united states",2024-02-22,13.18
1,REC-900952,12550,prince goût chocolat,lu,300.0,"algeria,belgium,france,french polynesia,german...",2024-11-13,1.50
2,REC-595965,6705,excellence noir prodigieux 90% cacao,"lindt excellence,lindt",100.0,"argelia,austria,bélgica,bulgaria,canadá,repúbl...",2025-02-02,12.99
3,REC-942653,8645,tonik,عربي,22.0,maroc,2024-12-11,14.82
4,REC-885998,14430,sésame,gerblé,230.0,"belgium, bulgaria, france, en:switzerland",2025-08-21,18.40


Now our dataset are ready for analysis. We have fill the missing and unformated values.